### 1. 환경 설정 및 상태 추적기

구글 드라이브를 연결하고, 5개의 압축 파일 경로와 학습 상태를 저장할 JSON 파일 경로를 설정합니다.

In [2]:
import os
import json
import glob
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive', force_remount=True)

# 2. 경로 설정
DATA_DIR = '/content/drive/MyDrive/UNet'
MODEL_DIR = '/content/drive/MyDrive/UNet_GAN_Upgraded' # 경로 변경

NOISE_ZIPS = glob.glob(f'{DATA_DIR}/Noise/TS_*.zip')
VOICE_ZIPS = [
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_01.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_02.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_03.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_04.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_05.zip'
]

CHECKPOINT_DIR = f'{MODEL_DIR}/Checkpoints'
STATE_FILE = f'{CHECKPOINT_DIR}/training_state.json'

if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r') as f:
            return json.load(f)
    return {"current_zip_idx": 0, "current_epoch": 0}

def save_state(zip_idx, epoch):
    state = {"current_zip_idx": zip_idx, "current_epoch": epoch}
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f)
    print(f"💾 상태 저장: [Zip {zip_idx+1}/{len(VOICE_ZIPS)}], [Epoch {epoch}]")

print(f"✅ 업그레이드 모델 전용 환경 설정 완료. (저장: {MODEL_DIR})")

Mounted at /content/drive
✅ 업그레이드 모델 전용 환경 설정 완료. (저장: /content/drive/MyDrive/UNet_GAN_Upgraded)


### 2. 모델 및 전처리 함수 정의 (Concat U-Net + ResNet)
기존 U-Net의 한계를 넘기 위해 스킵 커넥션을 Concat 방식으로 강화하고 잔차 블록을 추가한 구조입니다.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return F.leaky_relu(x + self.conv(x), 0.2)

class UpgradedUNet(nn.Module):
    def __init__(self):
        super(UpgradedUNet, self).__init__()
        # Encoder
        self.enc1 = nn.Sequential(nn.Conv2d(1, 64, 3, padding=1), nn.LeakyReLU(0.2))
        self.res1 = ResBlock(64)
        self.down1 = nn.Conv2d(64, 128, 4, stride=2, padding=1)

        self.res2 = ResBlock(128)
        self.down2 = nn.Conv2d(128, 256, 4, stride=2, padding=1)

        self.res3 = ResBlock(256)

        # Decoder
        self.up2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), ResBlock(128))

        self.up1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), ResBlock(64))

        self.final = nn.Conv2d(64, 1, 3, padding=1)

    def forward(self, x):
        e1 = self.res1(self.enc1(x))
        e2 = self.res2(F.leaky_relu(self.down1(e1), 0.2))
        e3 = self.res3(F.leaky_relu(self.down2(e2), 0.2))

        d2 = self.up2(e3)
        if d2.size() != e2.size(): d2 = F.interpolate(d2, size=e2.shape[2:])
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        if d1.size() != e1.size(): d1 = F.interpolate(d1, size=e1.shape[2:])
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        mask = torch.sigmoid(self.final(d1))
        return x * mask, mask

class UpgradedDiscriminator(nn.Module):
    def __init__(self):
        super(UpgradedDiscriminator, self).__init__()
        def conv_block(in_f, out_f, stride=2):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, 4, stride=stride, padding=1),
                nn.BatchNorm2d(out_f),
                nn.LeakyReLU(0.2, inplace=True)
            )
        self.model = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.model(x)

def get_spec(audio):
    window = torch.hann_window(512).to(audio.device)
    spec = torch.stft(audio.squeeze(1), n_fft=512, hop_length=160, window=window, return_complex=True, center=True)
    return torch.abs(spec), torch.angle(spec)

### 3. 디스크 관리 및 데이터 로더 정의

압축 해제 후 삭제를 통한 디스크 용량 관리 로직과 4초 이상(128KB) 데이터 필터링 로직을 포함합니다.

In [4]:
import glob
import zipfile
import shutil
import random
import numpy as np
import librosa
from torch.utils.data import Dataset, DataLoader

TEMP_VOICE_DIR = '/content/temp_voice'
TEMP_NOISE_DIR = '/content/temp_noise'

def manage_disk_and_extract(zip_path, extract_dir):
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
        print(f"🗑️ 기존 임시 폴더 삭제 완료: {extract_dir}")

    os.makedirs(extract_dir)
    print(f"📦 압축 해제 중... ({os.path.basename(zip_path)})")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("✅ 압축 해제 완료.")

class RobustDataset(Dataset):
    def __init__(self, voice_dir, noise_dir, samples=64000):
        all_voices = glob.glob(os.path.join(voice_dir, '**', '*.pcm'), recursive=True)
        self.voice_list = [f for f in all_voices if os.path.getsize(f) >= samples * 2]

        self.noise_list = glob.glob(os.path.join(noise_dir, '**', '*.mp3'), recursive=True)
        self.samples = samples
        print(f"📊 데이터 로드: 목소리 {len(self.voice_list)}개, 소음 {len(self.noise_list)}개 준비됨.")

    def load_pcm(self, path):
        try:
            with open(path, 'rb') as f:
                return np.frombuffer(f.read(), dtype=np.int16).astype(np.float32) / 32768.0
        except:
            return np.zeros(self.samples)

    def __len__(self):
        return len(self.voice_list)

    def __getitem__(self, idx):
        v_path = self.voice_list[idx]
        v_data = self.load_pcm(v_path)
        if len(v_data) < self.samples: return torch.zeros(1, self.samples), torch.zeros(1, self.samples)

        start = np.random.randint(0, len(v_data) - self.samples + 1)
        clean = v_data[start : start + self.samples]

        n_path = random.choice(self.noise_list)
        try:
            noise, _ = librosa.load(n_path, sr=16000, offset=8.5, duration=6.0)
            # 파일이 8.5초보다 짧은 경우 처음부터 다시 로드
            if len(noise) == 0:
                noise, _ = librosa.load(n_path, sr=16000, duration=6.0)
            # 여전히 비어있으면 0으로 채움
            if len(noise) == 0:
                noise = np.zeros(self.samples)
        except:
            noise = np.zeros(self.samples)

        if len(noise) < self.samples:
            noise = np.tile(noise, int(np.ceil(self.samples / len(noise))))
        start_n = np.random.randint(0, len(noise) - self.samples + 1)
        noise = noise[start_n : start_n + self.samples]

        snr = np.random.uniform(5, 15)
        clean_rms = np.sqrt(np.mean(clean**2) + 1e-9)
        noise_rms = np.sqrt(np.mean(noise**2) + 1e-9)
        noise = noise * (clean_rms / (10**(snr/20)) / noise_rms)

        noisy = clean + noise
        return torch.FloatTensor(noisy).unsqueeze(0), torch.FloatTensor(clean).unsqueeze(0)

print("디스크 관리자 및 데이터셋 클래스 준비 완료.")

디스크 관리자 및 데이터셋 클래스 준비 완료.


### 4. 메인 학습 루프 (Auto-Resume 및 LSGAN 적용)
경로를 upgraded_gan_model.pth로 설정하여 새 폴더에 저장되도록 했습니다.

In [6]:
import torch.optim as optim
from torch.cuda.amp import autocast
import torch
import zipfile
import gc
import os
import glob
from torch.utils.data import DataLoader

# 1. 배치 사이즈와 워커 설정
BATCH_SIZE = 32
NUM_WORKERS = 2 # 경고 해결을 위해 코랩 권장 수치인 2로 변경
EPOCHS_PER_ZIP = 10
LR_G = 1e-4
LR_D = 1e-5
LAMBDA_ADV = 0.01

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.empty_cache()
gc.collect()

# 2. 모델 및 스케일러 설정 (경고 해결: 최신 torch.amp.GradScaler 사용)
generator = UpgradedUNet().to(device)
discriminator = UpgradedDiscriminator().to(device)
scaler = torch.amp.GradScaler('cuda')

opt_G = optim.Adam(generator.parameters(), lr=LR_G)
opt_D = optim.Adam(discriminator.parameters(), lr=LR_D)

criterion_L1 = nn.L1Loss()
criterion_GAN = nn.MSELoss()

state = load_state()
start_zip_idx = state['current_zip_idx']
start_epoch = state['current_epoch']

latest_checkpoint = os.path.join(CHECKPOINT_DIR, 'upgraded_gan_model.pth')
if os.path.exists(latest_checkpoint):
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    generator.load_state_dict(checkpoint['generator_state_dict'])
    discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
    opt_G.load_state_dict(checkpoint['opt_G_state_dict'])
    opt_D.load_state_dict(checkpoint['opt_D_state_dict'])
    print(f"🔄 체크포인트 복구 완료.")

# 3. 소음 데이터 압축 해제 로직 복구 (비어있을 때만 해제)
if not os.path.exists(TEMP_NOISE_DIR) or len(glob.glob(f'{TEMP_NOISE_DIR}/**/*.mp3', recursive=True)) == 0:
    os.makedirs(TEMP_NOISE_DIR, exist_ok=True)
    for noise_zip in NOISE_ZIPS:
        print(f"📦 소음 압축 해제 중: {os.path.basename(noise_zip)}")
        with zipfile.ZipFile(noise_zip, 'r') as zip_ref:
            zip_ref.extractall(TEMP_NOISE_DIR)
    print("✅ 소음 데이터 압축 해제 완료.")

# 4. 메인 학습 루프
for zip_idx in range(start_zip_idx, len(VOICE_ZIPS)):
    manage_disk_and_extract(VOICE_ZIPS[zip_idx], TEMP_VOICE_DIR)
    dataset = RobustDataset(TEMP_VOICE_DIR, TEMP_NOISE_DIR)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

    for epoch in range(start_epoch, EPOCHS_PER_ZIP):
        generator.train()
        discriminator.train()

        for i, (noisy, clean) in enumerate(dataloader):
            noisy, clean = noisy.to(device, non_blocking=True), clean.to(device, non_blocking=True)

            with autocast():
                noisy_mag, _ = get_spec(noisy)
                clean_mag, _ = get_spec(clean)
                noisy_mag, clean_mag = noisy_mag.unsqueeze(1), clean_mag.unsqueeze(1)

                # [1] Discriminator 학습
                opt_D.zero_grad()
                real_preds = discriminator(clean_mag)
                denoised_mag, _ = generator(noisy_mag)
                fake_preds = discriminator(denoised_mag.detach())

                d_loss = 0.5 * (criterion_GAN(real_preds, torch.ones_like(real_preds)) +
                                criterion_GAN(fake_preds, torch.zeros_like(fake_preds)))

            scaler.scale(d_loss).backward()
            scaler.step(opt_D)

            # [2] Generator 학습
            with autocast():
                opt_G.zero_grad()
                g_loss_recon = criterion_L1(denoised_mag, clean_mag)
                fake_preds_for_G = discriminator(denoised_mag)
                g_loss_adv = criterion_GAN(fake_preds_for_G, torch.ones_like(fake_preds_for_G))
                g_loss = g_loss_recon + (LAMBDA_ADV * g_loss_adv)

            scaler.scale(g_loss).backward()
            scaler.step(opt_G)
            scaler.update()

            if (i+1) % 100 == 0:
                print(f"Step [{i+1}/{len(dataloader)}] D_Loss: {d_loss.item():.4f} | G_Loss: {g_loss.item():.4f}")

        # 에포크 단위 저장
        torch.save({
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'opt_G_state_dict': opt_G.state_dict(),
            'opt_D_state_dict': opt_D.state_dict(),
        }, latest_checkpoint)
        save_state(zip_idx, epoch + 1)

    start_epoch = 0
    del dataset, dataloader
    torch.cuda.empty_cache()
    gc.collect()

print("🎉 속도 최적화 버전 학습 완료!")

🔄 체크포인트 복구 완료.
📦 소음 압축 해제 중: TS_1.자연.zip
📦 소음 압축 해제 중: TS_10.기계 및 공구.zip
📦 소음 압축 해제 중: TS_2.무기.zip
📦 소음 압축 해제 중: TS_3.사람.zip
📦 소음 압축 해제 중: TS_4.동물.zip
📦 소음 압축 해제 중: TS_5.알람.zip
📦 소음 압축 해제 중: TS_6.물체.zip
📦 소음 압축 해제 중: TS_7.악기.zip
📦 소음 압축 해제 중: TS_8.군부대 운송수단.zip
📦 소음 압축 해제 중: TS_9.생활.zip
✅ 소음 데이터 압축 해제 완료.
📦 압축 해제 중... (KsponSpeech_04.zip)
✅ 압축 해제 완료.
📊 데이터 로드: 목소리 63749개, 소음 35848개 준비됨.
🗑️ 기존 임시 폴더 삭제 완료: /content/temp_voice
📦 압축 해제 중... (KsponSpeech_05.zip)
✅ 압축 해제 완료.
📊 데이터 로드: 목소리 65285개, 소음 35848개 준비됨.


/tmp/ipykernel_1578/3035296387.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1578/3035296387.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Step [100/2040] D_Loss: 0.2451 | G_Loss: 0.0134
Step [200/2040] D_Loss: 0.2367 | G_Loss: 0.0141
Step [300/2040] D_Loss: 0.2347 | G_Loss: 0.0134
Step [400/2040] D_Loss: 0.2328 | G_Loss: 0.0144
Step [500/2040] D_Loss: 0.2192 | G_Loss: 0.0148
Step [600/2040] D_Loss: 0.2422 | G_Loss: 0.0136
Step [700/2040] D_Loss: 0.2416 | G_Loss: 0.0115
Step [800/2040] D_Loss: 0.2332 | G_Loss: 0.0147
Step [900/2040] D_Loss: 0.2378 | G_Loss: 0.0123
Step [1000/2040] D_Loss: 0.2334 | G_Loss: 0.0142
Step [1100/2040] D_Loss: 0.2299 | G_Loss: 0.0165
Step [1200/2040] D_Loss: 0.2277 | G_Loss: 0.0143
Step [1300/2040] D_Loss: 0.2267 | G_Loss: 0.0141
Step [1400/2040] D_Loss: 0.2427 | G_Loss: 0.0143
Step [1500/2040] D_Loss: 0.2358 | G_Loss: 0.0135
Step [1600/2040] D_Loss: 0.2328 | G_Loss: 0.0149
Step [1700/2040] D_Loss: 0.2428 | G_Loss: 0.0118
Step [1800/2040] D_Loss: 0.2477 | G_Loss: 0.0119
Step [1900/2040] D_Loss: 0.2371 | G_Loss: 0.0134
Step [2000/2040] D_Loss: 0.2653 | G_Loss: 0.0120
💾 상태 저장: [Zip 5/5], [Epoch 1]

결과 검증 및 시각화

In [2]:
import os
import random
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive', force_remount=True)

# 2. 경로 설정 (Upgraded 전용 경로 및 단일 노이즈 압축파일)
DATA_DIR = '/content/drive/MyDrive/UNet'
MODEL_DIR = '/content/drive/MyDrive/UNet_GAN_Upgraded'
CHECKPOINT_DIR = f'{MODEL_DIR}/Checkpoints'
RESULT_DIR = f'{MODEL_DIR}/Test_Results'

if not os.path.exists(RESULT_DIR):
    os.makedirs(RESULT_DIR)

VOICE_ZIP = f'{DATA_DIR}/Korean_Voice/KsponSpeech_01.zip'
NOISE_ZIP = f'{DATA_DIR}/Clean_Noise_Zips/Cleaned_Noise_All.zip'

# 3. 업그레이드된 모델 및 전처리 함수 정의
class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return F.leaky_relu(x + self.conv(x), 0.2)

class UpgradedUNet(nn.Module):
    def __init__(self):
        super(UpgradedUNet, self).__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 64, 3, padding=1), nn.LeakyReLU(0.2))
        self.res1 = ResBlock(64)
        self.down1 = nn.Conv2d(64, 128, 4, stride=2, padding=1)
        self.res2 = ResBlock(128)
        self.down2 = nn.Conv2d(128, 256, 4, stride=2, padding=1)
        self.res3 = ResBlock(256)
        self.up2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), ResBlock(128))
        self.up1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), ResBlock(64))
        self.final = nn.Conv2d(64, 1, 3, padding=1)

    def forward(self, x):
        e1 = self.res1(self.enc1(x))
        e2 = self.res2(F.leaky_relu(self.down1(e1), 0.2))
        e3 = self.res3(F.leaky_relu(self.down2(e2), 0.2))
        d2 = self.up2(e3)
        if d2.size() != e2.size(): d2 = F.interpolate(d2, size=e2.shape[2:])
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        if d1.size() != e1.size(): d1 = F.interpolate(d1, size=e1.shape[2:])
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        mask = torch.sigmoid(self.final(d1))
        return x * mask, mask

def get_spec(audio):
    window = torch.hann_window(512).to(audio.device)
    spec = torch.stft(audio.squeeze(1), n_fft=512, hop_length=160, window=window, return_complex=True, center=True)
    return torch.abs(spec), torch.angle(spec)

def spec_to_wav(mag, phase):
    complex_spec = torch.polar(mag, phase)
    return torch.istft(complex_spec, n_fft=512, hop_length=160, center=True)

def get_random_test_samples(voice_zip, noise_zip, temp_dir='/content/temp_test'):
    if not os.path.exists(temp_dir): os.makedirs(temp_dir)
    with zipfile.ZipFile(voice_zip, 'r') as z:
        min_bytes = 16000 * 2 * 4
        valid_infos = [info for info in z.infolist() if info.filename.endswith('.pcm') and info.file_size >= min_bytes]
        target_info = random.choice(valid_infos)
        z.extract(target_info, temp_dir)
        voice_path = os.path.join(temp_dir, target_info.filename)

    with zipfile.ZipFile(noise_zip, 'r') as z:
        wav_files = [f for f in z.namelist() if f.endswith('.wav')]
        target_wav = random.choice(wav_files)
        z.extract(target_wav, temp_dir)
        noise_path = os.path.join(temp_dir, target_wav)
    return voice_path, noise_path

# 4. 검증 실행 (10회 반복)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
latest_checkpoint = os.path.join(CHECKPOINT_DIR, 'upgraded_gan_model.pth')

if not os.path.exists(latest_checkpoint):
    print(f"❌ 오류: {latest_checkpoint} 파일이 없습니다.")
else:
    print(f"✅ 모델 로드: {os.path.basename(latest_checkpoint)}")
    model = UpgradedUNet().to(device)
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['generator_state_dict'])
    model.eval()

    for i in range(10):
        print(f"\n--- 테스트 샘플 {i+1}/10 진행 중 ---")
        v_path, n_path = get_random_test_samples(VOICE_ZIP, NOISE_ZIP)

        with open(v_path, 'rb') as f:
            clean = np.frombuffer(f.read(), dtype=np.int16).astype(np.float32) / 32768.0

        try:
            # 전처리된 파일이므로 offset 없이 로드
            noise, _ = librosa.load(n_path, sr=16000, duration=6.0)
        except:
            noise = np.zeros(16000 * 4)

        sample_len = 16000 * 4
        clean = clean[:sample_len]
        noise = noise[:sample_len] if len(noise) >= sample_len else np.resize(noise, sample_len)
        noisy = clean + noise * 0.5

        with torch.no_grad():
            noisy_t = torch.FloatTensor(noisy).unsqueeze(0).unsqueeze(0)
            clean_t = torch.FloatTensor(clean).unsqueeze(0).unsqueeze(0)

            # 노이즈가 섞인 스펙트로그램
            mag, phase = get_spec(noisy_t)
            mag = mag.unsqueeze(1).to(device)

            # 원본(Clean) 스펙트로그램
            clean_mag, _ = get_spec(clean_t)

            # 모델 추론
            denoised_mag, mask = model(mag)
            denoised_wav = spec_to_wav(denoised_mag.squeeze(1).cpu(), phase)

        # 시각화 (4x1 서브플롯 구성)
        plt.figure(figsize=(12, 13))

        plt.subplot(4, 1, 1)
        plt.title("Noisy Input (Upgraded)")
        plt.imshow(torch.log1p(mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
        plt.colorbar()

        plt.subplot(4, 1, 2)
        plt.title("Predicted Mask (Upgraded)")
        plt.imshow(mask.squeeze().cpu().numpy(), aspect='auto', origin='lower', cmap='bone')
        plt.colorbar()

        plt.subplot(4, 1, 3)
        plt.title("Denoised Output (Upgraded)")
        plt.imshow(torch.log1p(denoised_mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
        plt.colorbar()

        plt.subplot(4, 1, 4)
        plt.title("Clean Target (Original Voice)")
        plt.imshow(torch.log1p(clean_mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
        plt.colorbar()

        plt.tight_layout()

        # 결과 저장
        save_img_path = os.path.join(RESULT_DIR, f'upgraded_result_spec_{i+1}.png')
        plt.savefig(save_img_path)
        plt.show()

        sf.write(os.path.join(RESULT_DIR, f'upgraded_test_noisy_{i+1}.wav'), noisy, 16000)
        sf.write(os.path.join(RESULT_DIR, f'upgraded_test_denoised_{i+1}.wav'), denoised_wav[0].numpy(), 16000)
        sf.write(os.path.join(RESULT_DIR, f'upgraded_test_clean_{i+1}.wav'), clean, 16000)
        print(f"🎧 저장 완료: 테스트 {i+1} 오디오 및 스펙트로그램 이미지")

    print(f"\n✅ 10개의 테스트 샘플 검증 및 저장이 모두 완료되었습니다. (위치: {RESULT_DIR})")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
!pip install -q pesq pystoi

import os
import random
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import math
from pesq import pesq
from pystoi import stoi
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive', force_remount=True)

# 2. 경로 설정
DATA_DIR = '/content/drive/MyDrive/UNet'
MODEL_DIR = '/content/drive/MyDrive/UNet_GAN_Upgraded'
CHECKPOINT_DIR = f'{MODEL_DIR}/Checkpoints'
RESULT_DIR = f'{MODEL_DIR}/Test_Results_Compare'

if not os.path.exists(RESULT_DIR):
    os.makedirs(RESULT_DIR)

VOICE_ZIP = f'{DATA_DIR}/Korean_Voice/KsponSpeech_01.zip'
NOISE_ZIP = f'{DATA_DIR}/Clean_Noise_Zips/Cleaned_Noise_All.zip'

# 3. 모델 및 함수 정의
class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return F.leaky_relu(x + self.conv(x), 0.2)

class UpgradedUNet(nn.Module):
    def __init__(self):
        super(UpgradedUNet, self).__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 64, 3, padding=1), nn.LeakyReLU(0.2))
        self.res1 = ResBlock(64)
        self.down1 = nn.Conv2d(64, 128, 4, stride=2, padding=1)
        self.res2 = ResBlock(128)
        self.down2 = nn.Conv2d(128, 256, 4, stride=2, padding=1)
        self.res3 = ResBlock(256)
        self.up2 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), ResBlock(128))
        self.up1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), ResBlock(64))
        self.final = nn.Conv2d(64, 1, 3, padding=1)

    def forward(self, x):
        e1 = self.res1(self.enc1(x))
        e2 = self.res2(F.leaky_relu(self.down1(e1), 0.2))
        e3 = self.res3(F.leaky_relu(self.down2(e2), 0.2))
        d2 = self.up2(e3)
        if d2.size() != e2.size(): d2 = F.interpolate(d2, size=e2.shape[2:])
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        if d1.size() != e1.size(): d1 = F.interpolate(d1, size=e1.shape[2:])
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        mask = torch.sigmoid(self.final(d1))
        return x * mask, mask

def get_spec(audio):
    window = torch.hann_window(512).to(audio.device)
    spec = torch.stft(audio.squeeze(1), n_fft=512, hop_length=160, window=window, return_complex=True, center=True)
    return torch.abs(spec), torch.angle(spec)

def spec_to_wav(mag, phase):
    complex_spec = torch.polar(mag, phase)
    return torch.istft(complex_spec, n_fft=512, hop_length=160, center=True)

def get_random_test_samples(voice_zip, noise_zip, temp_dir='/content/temp_test'):
    if not os.path.exists(temp_dir): os.makedirs(temp_dir)
    with zipfile.ZipFile(voice_zip, 'r') as z:
        min_bytes = 16000 * 2 * 4
        valid_infos = [info for info in z.infolist() if info.filename.endswith('.pcm') and info.file_size >= min_bytes]
        target_info = random.choice(valid_infos)
        z.extract(target_info, temp_dir)
        voice_path = os.path.join(temp_dir, target_info.filename)

    with zipfile.ZipFile(noise_zip, 'r') as z:
        wav_files = [f for f in z.namelist() if f.endswith('.wav')]
        target_wav = random.choice(wav_files)
        z.extract(target_wav, temp_dir)
        noise_path = os.path.join(temp_dir, target_wav)
    return voice_path, noise_path

def calculate_metrics(clean_wav, denoised_wav, sr=16000):
    min_len = min(len(clean_wav), len(denoised_wav))
    clean_wav = clean_wav[:min_len]
    denoised_wav = denoised_wav[:min_len]

    try:
        pesq_score = pesq(sr, clean_wav, denoised_wav, 'wb')
    except:
        pesq_score = 0.0

    stoi_score = stoi(clean_wav, denoised_wav, sr, extended=False)

    noise = clean_wav - denoised_wav
    signal_power = np.sum(clean_wav ** 2)
    noise_power = np.sum(noise ** 2)
    snr_score = 10 * math.log10(signal_power / (noise_power + 1e-9))

    return pesq_score, stoi_score, snr_score

# 4. 검증 실행 (10회 반복)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
latest_checkpoint = os.path.join(CHECKPOINT_DIR, 'upgraded_gan_model.pth')

if not os.path.exists(latest_checkpoint):
    print(f"❌ 오류: {latest_checkpoint} 파일이 없습니다.")
else:
    print(f"✅ 모델 로드: {os.path.basename(latest_checkpoint)}")
    model = UpgradedUNet().to(device)
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['generator_state_dict'])
    model.eval()

    for i in range(10):
        print(f"\n--- 테스트 샘플 {i+1}/10 진행 중 ---")
        v_path, n_path = get_random_test_samples(VOICE_ZIP, NOISE_ZIP)

        with open(v_path, 'rb') as f:
            clean = np.frombuffer(f.read(), dtype=np.int16).astype(np.float32) / 32768.0

        try:
            noise, _ = librosa.load(n_path, sr=16000, duration=6.0)
        except:
            noise = np.zeros(16000 * 4)

        sample_len = 16000 * 4
        clean = clean[:sample_len]
        noise = noise[:sample_len] if len(noise) >= sample_len else np.resize(noise, sample_len)
        noisy = clean + noise * 0.5

        with torch.no_grad():
            noisy_t = torch.FloatTensor(noisy).unsqueeze(0).unsqueeze(0)
            clean_t = torch.FloatTensor(clean).unsqueeze(0).unsqueeze(0)

            # 스펙트로그램 추출
            mag, phase = get_spec(noisy_t)
            mag = mag.unsqueeze(1).to(device)
            clean_mag, _ = get_spec(clean_t)

            # 모델 추론
            denoised_mag, mask = model(mag)
            denoised_wav = spec_to_wav(denoised_mag.squeeze(1).cpu(), phase)
            denoised_wav_np = denoised_wav[0].numpy()

        # 성능 지표 계산 및 출력
        pesq_val, stoi_val, snr_val = calculate_metrics(clean, denoised_wav_np)
        print(f"📊 [샘플 {i+1}] 성능 지표: PESQ: {pesq_val:.2f} | STOI: {stoi_val:.2f} | SNR: {snr_val:.2f}dB")

        # 시각화 (4x1 서브플롯 구성)
        plt.figure(figsize=(12, 13))

        plt.subplot(4, 1, 1)
        plt.title("Noisy Input (Upgraded)")
        plt.imshow(torch.log1p(mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
        plt.colorbar()

        plt.subplot(4, 1, 2)
        plt.title("Predicted Mask (Upgraded)")
        plt.imshow(mask.squeeze().cpu().numpy(), aspect='auto', origin='lower', cmap='bone')
        plt.colorbar()

        plt.subplot(4, 1, 3)
        plt.title("Denoised Output (Upgraded)")
        plt.imshow(torch.log1p(denoised_mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
        plt.colorbar()

        plt.subplot(4, 1, 4)
        plt.title("Clean Target (Original Voice)")
        plt.imshow(torch.log1p(clean_mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
        plt.colorbar()

        plt.tight_layout()

        # 결과 저장
        save_img_path = os.path.join(RESULT_DIR, f'upgraded_result_spec_{i+1}.png')
        plt.savefig(save_img_path)
        plt.show()

        sf.write(os.path.join(RESULT_DIR, f'upgraded_test_noisy_{i+1}.wav'), noisy, 16000)
        sf.write(os.path.join(RESULT_DIR, f'upgraded_test_denoised_{i+1}.wav'), denoised_wav_np, 16000)
        sf.write(os.path.join(RESULT_DIR, f'upgraded_test_clean_{i+1}.wav'), clean, 16000)
        print(f"🎧 저장 완료: 테스트 {i+1} 오디오 및 스펙트로그램 이미지")

    print(f"\n✅ 10개의 테스트 샘플 검증 및 저장이 모두 완료되었습니다. (위치: {RESULT_DIR})")